In [1]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00


In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import numpy as np

# ==================================================
# PARAMETERS
# ==================================================
n_bits = 100
noise_level = 0.05
eve_attack = 0.2

simulator = Aer.get_backend('aer_simulator')

# ==================================================
# FUNCTIONS
# ==================================================

def generate_bits(n):
    return np.random.randint(0, 2, n)

def generate_bases(n):
    return np.random.randint(0, 2, n)  # 0 = Z, 1 = X

# ==================================================
# ENCODE USING QUANTUM CIRCUIT
# ==================================================
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)

    if bit == 1:
        qc.x(0)

    if basis == 1:  # X basis
        qc.h(0)

    return qc

# ==================================================
# EVE INTERCEPTION (QUANTUM)
# ==================================================
def eve_intercept(qc, attack_prob):
    if np.random.rand() < attack_prob:
        eve_basis = np.random.randint(0, 2)

        if eve_basis == 1:
            qc.h(0)

        qc.measure(0, 0)

        compiled = transpile(qc, simulator)
        result = simulator.run(compiled, shots=1).result()
        measured = int(list(result.get_counts().keys())[0])

        # Re-prepare qubit
        new_qc = QuantumCircuit(1, 1)
        if measured == 1:
            new_qc.x(0)
        if eve_basis == 1:
            new_qc.h(0)

        return new_qc
    return qc

# ==================================================
# NOISE
# ==================================================
def apply_noise(qc, noise_level):
    if np.random.rand() < noise_level:
        qc.x(0)
    return qc

# ==================================================
# BOB MEASUREMENT
# ==================================================
def measure_qubit(qc, basis):
    if basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()

    return int(list(result.get_counts().keys())[0])

# ==================================================
# MAIN SIMULATION
# ==================================================

alice_bits = generate_bits(n_bits)
alice_bases = generate_bases(n_bits)
bob_bases = generate_bases(n_bits)

bob_results = []

for i in range(n_bits):

    qc = encode_qubit(alice_bits[i], alice_bases[i])

    qc = eve_intercept(qc, eve_attack)

    qc = apply_noise(qc, noise_level)

    result = measure_qubit(qc, bob_bases[i])

    bob_results.append(result)

# ==================================================
# SIFTING
# ==================================================
sifted_a = []
sifted_b = []

for i in range(n_bits):
    if alice_bases[i] == bob_bases[i]:
        sifted_a.append(alice_bits[i])
        sifted_b.append(bob_results[i])

# ==================================================
# ERROR RATE
# ==================================================
sifted_a = np.array(sifted_a)
sifted_b = np.array(sifted_b)

error_rate = np.sum(sifted_a != sifted_b) / len(sifted_a)

print("Error Rate:", error_rate)

# ==================================================
# DETECTION
# ==================================================
if error_rate > noise_level + 0.02:
    print("⚠️ Eavesdropping detected!")
else:
    print("✅ Channel seems normal")

Error Rate: 0.07692307692307693
⚠️ Eavesdropping detected!
